In [2]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


In [4]:
class JokeState(TypedDict):
    joke:str
    topic:str
    explanation:str

In [5]:
def generate_joke(state:JokeState):
    prompt=f'generate a joke on the topic {state["topic"]}'
    response=llm.invoke(prompt).content
    return {"joke": response}

In [6]:
def generate_explanation(state:JokeState):
    prompt=f'write the explanation for the joke: {state["joke"]}'
    response=llm.invoke(prompt).content
    return {"explanation": response}

In [7]:
graph=StateGraph(JokeState)
graph.add_node("generate_joke",generate_joke)
graph.add_node("generate_explanation", generate_explanation)
graph.add_edge(START,"generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation",END)


checkpointer=InMemorySaver()
workflow=graph.compile(checkpointer=checkpointer)

In [8]:
config1={"configurable":{"thread_id": "1"}}
workflow.invoke({"topic":"programming"},config=config1)

{'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs!',
 'topic': 'programming',
 'explanation': 'This joke is a classic example of wordplay, specifically a pun, that relies on the double meaning of the word "bugs."\n\nHere\'s the breakdown:\n\n1.  **The Literal Meaning of "Bugs":** In the real world, insects (bugs) are often attracted to light sources, especially at night. If you have a bright screen in a dark room, you might notice moths or other small insects flying towards it.\n\n2.  **The Programming Meaning of "Bugs":** In the context of programming, a "bug" is an error, flaw, or defect in software that causes it to produce an incorrect or unexpected result, or to behave in unintended ways. Programmers spend a significant amount of their time finding and fixing these "bugs."\n\n**The Punchline\'s Double Whammy:**\n\nThe joke combines these two meanings. When a programmer says they prefer dark mode "because light attracts bugs," they are humorously implyin

In [ ]:
workflow.get_state(config1)

In [ ]:
list(workflow.get_state_history(config=config1))